In [ ]:
import numpy as np
import cv2

camera_matrix = np.load("camera_matrix.npy")
dist_coeffs = np.load("dist_coeffs.npy")
R_shelf = np.load("R_shelf.npy")
T_shelf = np.load("T_shelf.npy")

def pixel_to_shelf(cx, cy, shelf_z=0):
    """Convert YOLO pixel (cx, cy) to real-world (X, Y, Z) on the Z=shelf plane."""

    # Undistort pixel to normalized camera ray
    undistorted = cv2.undistortPoints(
        np.array([[[cx, cy]]], dtype=np.float32),
        camera_matrix,
        dist_coeffs
    )[0][0]

    Xn, Yn = undistorted
    ray_cam = np.array([Xn, Yn, 1.0])  # camera ray

    # Camera origin in shelf coordinates
    C_shelf = -R_shelf.T @ T_shelf

    # Ray in shelf coordinates
    ray_shelf = R_shelf.T @ ray_cam

    # Solve intersection with shelf plane Z = shelf_z
    t = (shelf_z - C_shelf[2]) / ray_shelf[2]

    P = C_shelf + t * ray_shelf  # (X, Y, Z) position of object

    return P[0], P[1], P[2]
